In [1]:
import pandas as pd
import numpy as np
import mglearn
import matplotlib.pyplot as plt

In [2]:
#make_ppelineによる簡易まパイプライン作成
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC

#標準の構文
pipe_long = Pipeline([
     ("scaler",MinMaxScaler()),
     ("svm",SVC(C=100))
])

#短縮構文
pipe_short = make_pipeline(
    MinMaxScaler(),
    SVC(C=100)
)

#パイプラインオブジェクトpipe_longとpipe_shortは全く同じように動作するが、pipe_shortの各ステップは自動的に生成された名前がついている
#stepds属性を見ると各ステップの名前がわかる
print("Pipeline steps:\n{}".format(pipe_short.steps))

Pipeline steps:
[('minmaxscaler', MinMaxScaler()), ('svc', SVC(C=100))]


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pipe = make_pipeline(
    StandardScaler(),
    PCA(n_components=2),
    StandardScaler(),
)

#一つ目のStandardScalerの名前は自動的に"standardscaler-1"となる
#二つ目のStandardScalerの名前は自動的に"standardscaler-2"となる
#このような場合には、明示的に名前を各ステップに与えたほうがいいだろう
#その場合は、Pipelineを使うといい
print("Pipeline steps:\n{}".format(pipe.steps))

Pipeline steps:
[('standardscaler-1', StandardScaler()), ('pca', PCA(n_components=2)), ('standardscaler-2', StandardScaler())]


In [6]:
#パイプラインに含まれる各ステップの属性を見たいときは、named_steps属性を使う
#これは、ステップ名とEstimatorオブジェクトの辞書を返す

from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer()

pipe.fit(cancer.data)

#"pca"ステップ第２主成分を取り出す
components = pipe.named_steps["pca"].components_
print("PCA components shape: {}".format(components.shape))

PCA components shape: (2, 30)


In [9]:
#GridSearchCV内のパイプラインの属性へのアクセス
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=10000)
)

param_grid = {
    "logisticregression__C":[0.001,0.01,0.1,1,10,100]
}

x_trainval,x_test,y_trainval,y_test = train_test_split(
    cancer.data,
    cancer.target,
    random_state=4
)

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=5
)

grid.fit(x_trainval,y_trainval)

print("Best estimator:\n{}".format(grid.best_estimator_))

Best estimator:
Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(C=1, max_iter=10000))])


In [10]:
print("Logistic Regression step:\n{}".format(grid.best_estimator_.named_steps["logisticregression"]))

Logistic Regression step:
LogisticRegression(C=1, max_iter=10000)


In [13]:
print("Logistic Regression coefficients:\n{}".format(grid.best_estimator_.named_steps["logisticregression"].coef_))

Logistic Regression coefficients:
[[-0.4475566  -0.34609376 -0.41703843 -0.52889408 -0.15784407  0.60271339
  -0.71771325 -0.78367478  0.04847448  0.27478533 -1.29504052  0.05314385
  -0.69103766 -0.91925087 -0.14791795  0.46138699 -0.1264859  -0.10289486
   0.42812714  0.71492797 -1.08532414 -1.09273614 -0.85133685 -1.04104568
  -0.72839683  0.07656216 -0.83641023 -0.64928603 -0.6491432  -0.42968125]]
